# Практическая работа № 3 — Пайплайн предобработки
**Вариант A**

Разделение на train/test выполняется до обучения преобразований, чтобы избежать утечки данных.

In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

candidates = [Path("data/tips_sample.csv"), Path("../data/tips_sample.csv")]
data_path = next((p for p in candidates if p.exists()), None)
df = pd.read_csv(data_path)
df.head()


,total_bill,size,day,time,tip
0,24.44,1,Fri,Lunch,4.62
1,13.68,3,Sat,Dinner,2.99
2,28.00,2,Sun,Dinner,5.01
3,NaN,4,Sun,Lunch,5.85
4,6.39,4,Fri,Dinner,2.13


In [2]:
# Для демонстрации предсказываем tip по остальным признакам
X = df.drop(columns=["tip"])
y = df["tip"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
numeric_features = ["total_bill", "size"]
categorical_features = ["day", "time"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)
print("Train shape:", X_train_processed.shape)
print("Test shape:", X_test_processed.shape)


Train shape: (128, 8)
Test shape: (32, 8)


## Краткое объяснение
- Числовые пропуски заполнены медианой, менее чувствительной к выбросам.
- Категориальные пропуски заполнены наиболее частой категорией.
- Категории преобразованы через OneHotEncoder.
- StandardScaler масштабирует числовые признаки.
- `fit_transform` используется только на train; test проходит через `transform`.